In [ ]:
import logging
import os
import json
import sys

notebook_dir = os.getcwd()
project_dir = os.path.dirname(os.path.dirname(notebook_dir))

if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

print(f'Notebook dir: {notebook_dir}\nProject dir: {project_dir}.')
import force_regression.plotting.mu_plot as mu_plot
from force_regression.config.dataconfig import DataConfig
import force_regression.utils.functions as fn
from force_regression.models.linear_regression import ConventionalRegressionOnMu
from configs.constants import *
logging.getLogger().setLevel(logging.INFO)
%load_ext autoreload
%autoreload 2

In [ ]:
config_path = os.path.join(project_dir, 'configs/config.json')

try:
    with open(config_path, 'r') as config_file:
        config = json.load(config_file)
        root_dir = config['root_dir']
        root_results_dir = config['root_results_dir']
        subject_mappings = config['subject_mappings']
        print(f'Root directory from config: {root_dir}')
except FileNotFoundError:
    print(f"Error: 'config.json' not found in {config_path}")
except json.JSONDecodeError:
    print("Error: 'config.json' is not a valid JSON file.")
except KeyError:
    print("Error: 'root_dir' not found in 'config.json'.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

# Define parameters

In [ ]:
# Parameter cell for Papermill
subj = "S1"
mvc = 15
emg_type = 'intra'
sign_mvc = 1

model_type = 'linear'
overlap_in_perc = 50          # percentage overlap between the analysis windows
win_size_in_sec= 0.08         # window size in seconds
select_dir = True   # parameter to select direction of force to be used for regression. If False, both directions are used.
load_multi =[mvc]   #[mvc] the default is set to the given MVC value. Another option is a list of MVC

sweep_win_size=False   # Flag to sweep the window size. If True place results in a subfolder
segment_hold = False
load_regression_data_from_file = True
post_process= True
shuffle_fingers_seed = 10

In [ ]:
config = DataConfig(root_dir=root_dir, 
                    root_results_dir = root_results_dir,
                    subject=fn.reverse_remap(subj, subject_mappings),
                    task_type="Trap", finger_type="Individual", day="Day 1", 
                    mvc=mvc, emg_type=emg_type, f_samp=10240,
                    subj_map=subject_mappings,
                    segment_hold=segment_hold, 
                    verbose=True,
                    common_only=True, 
                    images=False, 
                    time_to_cut=1,
                    load_multi=load_multi,
                    convreg_temp_data_dir='regression_data',
                    figs_dir = 'figures')

output_figures_dir = os.path.join(config.root_results_dir,
                                  config.figs_dir)

output_figures_dir


# Load MUs and force dataframes

In [ ]:
mu_reg = ConventionalRegressionOnMu(model_type,
                                    config,
                                    emg_type,
                                    regression_data_parent_dir=config.convreg_temp_data_path,
                                    overlap_in_perc=overlap_in_perc,
                                    window_size_in_sec=win_size_in_sec,
                                    post_process=post_process,
                                    load_regression_data_from_file=load_regression_data_from_file,
                                    shuffle_fingers_seed=shuffle_fingers_seed

                                    )

In [ ]:
# to load the saved mu_df and force df
mu_reg.create_mu_regression_df()

In [ ]:
mu_reg.data_config.force_cols_list

In [ ]:
force_df = mu_reg.force_df
force_df[force_df[FING_NAME_COL]=='thumb']
temp_df = force_df[force_df[FING_NAME_COL]=='thumb']
temp_df[(temp_df[FING_DIR] == 'flex') & (temp_df[REP_ID]==0)]

In [ ]:
for plot_dir in [['flex']]:
    mu_plot.raster_plot_sep_rep_inset(mu_reg.mu_df_sorted, config, mu_reg.force_df, 
                                    plot_direction=plot_dir,
                                    plot_common_only=True, 
                                    save_fig=False,
                                    rep_list=[2],
                                    max_ylim=55,  #flex: 55, ext:rep1:80
                                    plot_all_forces=True,
                                    same_mus_color_per_task=False,
                                    output_figures_dir=output_figures_dir)